# **BizFlow360 Logistic Regression Model**
* **By**: Edusei Mikel
* **Date**: 6th August, 2026

# Path Configuration (Synthetic Pipeline)
**What this cell does:** Points all model, preprocessor, and metric saves to the `on_synthetic_data` subfolders, keeping synthetic artifacts cleanly separated from the real KNBS artifacts.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

# Loading Synthetic MSME Data
df = pd.read_csv("~/BizFlow360/data_eda/synthetic/synthetic_msme_data.csv")

print(f"✅ Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

✅ Dataset loaded successfully!
Shape: (5000, 16)


,business_id,county,sector,business_age_months,employees,monthly_revenue_kes,monthly_expenses_kes,total_assets_kes,total_liabilities_kes,loan_amount_kes,mpesa_volume_kes,expense_ratio,debt_to_asset_ratio,age_risk_factor,mpesa_dependency,distress_label
0,MSME_00000,Mombasa,Digital Commerce,103,15,69956.64,57239.43,54653.04,33300.95,74297.39,47382.18,0.818201,0.609304,0.009615,0.677298,1
1,MSME_00001,Kisumu,Construction,52,1,204523.61,134370.03,1084686.49,575481.67,0.00,139924.33,0.656987,0.530551,0.018868,0.684144,0
2,MSME_00002,Thika,Manufacturing,93,5,164160.29,174074.09,188031.92,110086.08,0.00,131168.13,1.060385,0.585462,0.010638,0.799020,1
3,MSME_00003,Nakuru,Manufacturing,15,10,350911.73,316756.20,279774.52,180768.18,44524.97,219436.17,0.902664,0.646119,0.062500,0.625330,1
4,MSME_00004,Nairobi,Digital Commerce,107,1,119510.33,117123.15,5870586.02,1401686.01,0.00,104834.82,0.980017,0.238764,0.009259,0.877196,0


**Preprocessing (Encodiing Categorical Variables**)

In [2]:
from sklearn.preprocessing import LabelEncoder

# Initializing Label Encoders
le_county = LabelEncoder()
le_sector = LabelEncoder()

# Encode 'county' and 'sector' numbers
df['county_encoded'] = le_county.fit_transform(df['county'])
df['sector_encoded'] = le_sector.fit_transform(df['sector'])

# Define our Features (X) and Target (y)
features = [
    'business_age_months', 'employees',
    'monthly_revenue_kes', 'monthly_expenses_kes',
    'total_assets_kes', 'total_liabilities_kes',
    'loan_amount_kes', 'mpesa_volume_kes', 
    'county_encoded', 'sector_encoded'
]

X = df[features]
y = df['distress_label']

print("✅ Preprocessing complete. Features and target defined.")

✅ Preprocessing complete. Features and target defined.


**Train/Test Split and Scaling**

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data: 80% training, 20% testing (Stratified to keep balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Data split and scaled successfully.")

✅ Data split and scaled successfully.


**Train the Baseline Model**

In [4]:
from sklearn.linear_model import LogisticRegression

# Initialize and train the Logistic Regression model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

print("✅ Logistic Regression Baselinr model trained successfully.")

✅ Logistic Regression Baselinr model trained successfully.


**Model Evaluation**

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# Make predictions
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("="*40)
print(" BASELINE MODEL METRICS")
print("="*40)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("="*40)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

 BASELINE MODEL METRICS
Accuracy:  0.7390
Precision: 0.7294
Recall:    0.7600
F1-Score:  0.7444
ROC-AUC:   0.7895

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.72      0.73       500
           1       0.73      0.76      0.74       500

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000



**Saving the Model and Preprocessors**

In [7]:
# Create directories if they don't exist
os.makedirs('../models/trained/on_synthetic_data', exist_ok=True)
os.makedirs('../models/preprocessing/on_synthetic_data', exist_ok=True)

# Save the model, scaler, and encoders
joblib.dump(model, '../models/trained/on_synthetic_data/logistic_regression_baseline.joblib')
joblib.dump(scaler, '../models/preprocessing/on_synthetic_data/scaler.joblib')
joblib.dump(le_county, '../models/preprocessing/on_synthetic_data/le_county.joblib')
joblib.dump(le_sector, '../models/preprocessing/on_synthetic_data/le_sector.joblib')

print("✅ Model and preprocessing files saved to ml_models/models/on_synthetic_data !")

✅ Model and preprocessing files saved to ml_models/models/on_synthetic_data !
